# ESI Integrate Demo

Look up antiderivatives from the Exhaustive Symbolic Integration database.

Given an integrand $f(x)$, ESI finds $F(x)$ such that $F'(x) = f(x)$ by matching
the numerical fingerprint of $f(x)$ against a pre-computed database of derivatives.

## 1. Load the database

This only needs to be done once per session. It takes a few seconds.

In [1]:
from esi_integrate import ESIDatabase, find_databases

db = ESIDatabase()
for pkl in find_databases('.'):
    db.load(pkl, verbose=True)

print(f"\nReady: {db.n_integrands} integrands from {', '.join(db.bases_loaded)}")

Loading ext_log_maths_k6... 5401 integrands (0.0s)
Loading ext_log_maths_k8... 198411 integrands (1.0s)
Loading ext_maths_k5... 0 integrands (0.0s)
Loading ext_maths_k7... 32 integrands (0.0s)
Loading ext_maths_k8... 728 integrands (0.3s)
Loading k5... 0 integrands (0.0s)
Loading k7... 0 integrands (0.0s)
Loading trig_maths_k8... 127347 integrands (0.4s)

Ready: 331919 integrands from ext_log_maths_k6, ext_log_maths_k8, ext_maths_k5, ext_maths_k7, ext_maths_k8, k5, k7, trig_maths_k8


## 2. Look up integrals

Use `db.lookup(expr)` with a SymPy expression, or the helper `integrate(expr_str)` with a string.

In [2]:
from esi_integrate import parse_input

def integrate(expr_str, show_all=False):
    """Look up an integral from the ESI database."""
    expr = parse_input(expr_str)
    if expr is None:
        print(f"Could not parse: {expr_str}")
        return None
    
    result = db.lookup(expr)
    if result is None:
        print(f"∫ {expr} dx  →  Not found")
        return None
    
    simplest_k, simplest_eq = result[0]
    print(f"∫ {expr} dx  =  {simplest_eq}  +  C    (k={simplest_k}, {len(result)} representations)")
    
    if show_all:
        n_show = min(10, len(result))
        for k, eq in result[:n_show]:
            print(f"  [k={k}] {eq}")
        if len(result) > n_show:
            print(f"  ... and {len(result) - n_show} more")
    
    return simplest_eq

In [3]:
# Basic examples
integrate("1/x")
integrate("exp(x)")
integrate("2*x")
integrate("sin(x)")

∫ 1/x dx  =  log(x)  +  C    (k=4, 680 representations)
∫ exp(x) dx  =  exp(x)  +  C    (k=4, 280 representations)
∫ 2*x dx  =  x**2  +  C    (k=3, 355 representations)
∫ sin(x) dx  =  a0 - cos(x)  +  C    (k=4, 181 representations)


'a0 - cos(x)'

In [4]:
# x^x and its relatives
integrate("x**x * (log(x) + 1)")
integrate("x**x * (log(x) + 1) * log(x)")

∫ x**x*(log(x) + 1) dx  =  pow(x,x)  +  C    (k=3, 139 representations)
∫ x**x*(log(x) + 1)*log(x) dx  →  Not found


In [5]:
# Known non-elementary: should return 'Not found'
integrate("exp(x**2)")
integrate("exp(exp(x))")
integrate("exp(x)/x")

∫ exp(x**2) dx  →  Not found
∫ exp(exp(x)) dx  →  Not found
∫ exp(x)/x dx  →  Not found


In [6]:
# Show all representations for 1/x
integrate("1/x", show_all=True)

∫ 1/x dx  =  log(x)  +  C    (k=4, 680 representations)
  [k=4] log(x)
  [k=4] log(2*x)
  [k=4] a0 + log(x)
  [k=4] log(x*Abs(1/a0))
  [k=5] log(x/2)
  [k=5] a0 - log(1/x)
  [k=5] log(x/Abs(a0))
  [k=5] -log(Abs(a0)/x)
  [k=5] log(x) - Abs(a0)
  [k=5] log(x) + Abs(a0)
  ... and 670 more


'log(x)'

## 3. Direct SymPy interface

You can also pass SymPy expressions directly to `db.lookup()`.

In [7]:
import sympy
x = sympy.Symbol('x', positive=True)

# Build an expression programmatically
f = sympy.exp(x) * sympy.sqrt(sympy.log(x))
result = db.lookup(f)

if result:
    k, F = result[0]
    print(f"∫ {f} dx  =  {F}  +  C   (k={k})")
else:
    print(f"∫ {f} dx  →  Not found")

∫ exp(x)*sqrt(log(x)) dx  →  Not found
